# 01 · FCN / U-Net on Oxford-IIIT Pet —— 从分类到逐像素

**家族位置**：`03_CNN_Segmentation_Detection` 第 1 站（分割 ★重点）。

**与上一站的衔接**：02 家族在 32×32 上把分类主干（LeNet→ConvNeXt）走完，本章把同一套卷积思想搬到**逐像素任务**：图像分类只给一图一签，语义分割每像素一个签。数据集不用 CIFAR，改用带 trimap 掩码的 **Oxford-IIIT Pet**（torchvision 自带，前景/背景/边界三类）——真正的分割数据形态。

**学习目标**
1. 分类 vs 检测 vs 分割的 supervision 粒度与评估体系（mIoU/Dice vs mAP）
2. U-Net 的编码器-解码器 + 跳跃连接机制（为什么分割离不开 skips）
3. FCN（无 skip）vs U-Net 的受控对照：唯一变量就是跳跃连接
4. 分割专用损失与指标：CE + Dice 联合、IoU 的逐类含义

## 1. 原理：为什么分割需要“跳跃”

### 通俗理解

**一句话**：分类把整张图压缩成一个标签，分割则要把每个像素的标签都吐出来——下采样会丢细节，上采样只会模糊补。U-Net 的**跳跃连接**把编码器高分辨率的“原图细节”直接抄给解码器，让网络既看得懂“这是什么”（深层语义）也记得“在哪儿”（浅层位置）。

**比喻**：抄地图——编码器像把地图越折越小只记要点（语义），解码器要重新展开。FCN 只靠记忆展开，细节全糊；U-Net 每折一次留一张小抄（skip），展开时对照着描，边界自然清晰。

### 结构账

```
FCN（基线）： enc1→pool→enc2→pool→enc3→pool→bottleneck → deconv×3 → refine → head   (无 skips)
U-Net：       enc1 ─┐   enc2 ─┐   enc3 ─┐                                   
                 pool     pool     pool                                    
                 bottleneck → deconv ┘cat → dec3 → deconv ┘cat → dec2 → deconv ┘cat → dec1 → head
```

两网**共享编码器容量**（32→64→128→256），唯一区别：U-Net 的 `cat([up, enc])` vs FCN 的纯上采样——消融的本体。

### 指标

- **mIoU** = 每类 IoU 求平均；IoU = 交/并（`pred∩gt / pred∪gt`），边界错一圈 IoU 就掉，极敏感
- **Dice** = 2·交/(pred+gt)，与 IoU 单调但对小目标更友好；做 loss 时用 soft 版，可导

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import CLASS_NAMES, IMG_SIZE, PetSegDataset, denorm, load_pet_segmentation
from common.engine import fit_seg, miou
from common.models import FCNMini, UNetMini, seg_flops
from common.utils import count_params, set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torch:", torch.__version__)

## 2. 数据：Oxford-IIIT Pet（trimap 三类）

37 类宠物品种但分割标签只有 3 类（0=背景 1=前景 2=边界），边界类最薄（<5% 像素）——天生类别不均衡，Dice 的主场。

In [ ]:
Xtr, ytr, Xva, yva = load_pet_segmentation(str(ROOT / "data"), train_n=500, val_n=200)
tr_dist = [(ytr==c).sum().item()/ytr.numel() for c in range(3)]
va_dist = [(yva==c).sum().item()/yva.numel() for c in range(3)]
print(f"训练 {Xtr.shape} {ytr.shape}  类别分布 train {[round(v,3) for v in tr_dist]}")
print(f"验证 {Xva.shape} {yva.shape}  类别分布 val   {[round(v,3) for v in va_dist]}")

# fig0：4 张样本 + 掩码叠加
cmap = plt.get_cmap("tab10", 3)
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    img = denorm(Xtr[i]).permute(1,2,0).numpy()
    m = ytr[i].numpy()
    axes[0, i].imshow(img); axes[0, i].axis("off"); axes[0, i].set_title(f"image {i}")
    axes[1, i].imshow(m, cmap="tab10", vmin=0, vmax=2); axes[1, i].axis("off")
    axes[1, i].set_title(f"mask {CLASS_NAMES[int(m[64,64])]}")
# colorbar 代替图例
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
plt.colorbar(plt.cm.ScalarMappable(cmap=cmap), cax=cbar_ax, ticks=[0.33, 1, 1.66])
cbar_ax.set_yticklabels(CLASS_NAMES, fontsize=9)
plt.suptitle("Oxford-IIIT Pet：原图 / trimap(背景/前景/边界)", fontsize=11)
plt.savefig(FIGS / "fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. 模型定稿：两网唯一差跳跃

参数/MACs 全家福 + FCN vs U-Net 分支说明。

In [ ]:
for name, cls in [("UNetMini", UNetMini), ("FCNMini", FCNMini)]:
    m = cls()
    print(f"{name:10s} 参数量={count_params(m):>7,} | MACs={seg_flops(m)/1e6:6.1f}M | in 3×128×128 → out {tuple(m(torch.randn(1,3,128,128)).shape)}")
print("\n对照设计：共享 enc 32→64→128→256，FCN 删三处 cat([up,enc])，其余容量一致。")

## 4. 主实验：FCN vs U-Net（约 12 分钟 CPU）

**协议**：500 train / 200 val、128×128、CE+Dice 联合损失、Adam 1e-3、batch 8、6 epochs、seed 0。两网同数据同优化同轮数——唯一变量是跳跃连接。

In [ ]:
EPOCHS = 6
tr = DataLoader(PetSegDataset(Xtr, ytr), batch_size=8, shuffle=True)
va = DataLoader(PetSegDataset(Xva, yva), batch_size=8)

results, models = {}, {}
for name, Cls in [("FCN", FCNMini), ("U-Net", UNetMini)]:
    set_seed(0)
    m = Cls()
    hist = fit_seg(m, tr, va, epochs=EPOCHS, lr=1e-3, device=DEVICE, verbose=True)
    results[name] = hist; models[name] = m
    print(f"{name:6s} final mIoU={hist['val_miou'][-1]:.4f} | Dice={hist['val_dice'][-1]:.4f}", flush=True)

In [ ]:
colors = {"FCN": "#C44E52", "U-Net": "#4C72B0"}
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for name, c in colors.items():
    axes[0].plot(results[name]["val_miou"], marker="o", ms=4, label=name, color=c)
    axes[1].plot(results[name]["val_dice"], marker="o", ms=4, label=name, color=c)
    axes[2].plot(results[name]["val_loss"], marker="o", ms=4, label=name, color=c)
axes[0].set_title("val mIoU"); axes[1].set_title("val Dice"); axes[2].set_title("val loss (CE+Dice)")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend()
plt.suptitle("FCN vs U-Net：跳跃连接的增益", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：终局柱状
fig, ax = plt.subplots(figsize=(6, 4))
names = ["FCN", "U-Net"]
mi = [results[n]["val_miou"][-1] for n in names]
bars = ax.bar(names, mi, color=[colors[n] for n in names])
for b, v in zip(bars, mi):
    ax.text(b.get_x()+b.get_width()/2, v, f"{v:.4f}", ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, 0.95); ax.set_ylabel("mIoU"); ax.set_title(f"终局 mIoU（Δ={mi[1]-mi[0]:+.4f}）")
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. 预测可视化：边界谁更清

同一批 4 张验证图，各列：原图 / 真实掩码 / FCN 预测 / U-Net 预测——直观看跳跃对边界的修复。

In [ ]:
idx = [5, 12, 20, 42]
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for r, j in enumerate(idx):
    img = Xva[j]
    gt = yva[j]
    with torch.no_grad():
        pf = models["FCN"](img.unsqueeze(0).to(DEVICE)).argmax(1)[0].cpu()
        pu = models["U-Net"](img.unsqueeze(0).to(DEVICE)).argmax(1)[0].cpu()
    for c, (mat, ttl) in enumerate([(img, "image"), (gt, "GT"), (pf, "FCN pred"), (pu, "U-Net pred")]):
        ax = axes[r, c]
        if c == 0:
            ax.imshow(denorm(mat).permute(1,2,0).numpy()); ax.set_title(ttl, fontsize=9)
        else:
            ax.imshow(mat.numpy(), cmap="tab10", vmin=0, vmax=2); ax.set_title(ttl, fontsize=9)
        ax.axis("off")
        if c == 0:
            ax.set_ylabel(f"#{j}", fontsize=8)
plt.suptitle("分割预测对比：U-Net 边界更贴合（skip 的价值）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig3_preds.png", dpi=150, bbox_inches="tight")
plt.show()

# 逐类 IoU 柱状（冠军 U-Net）
from common.engine import miou as _miou
with torch.no_grad():
    logits = models["U-Net"](Xva.to(DEVICE))
    pred = logits.argmax(1).cpu()
ious = {}
for c, name in enumerate(CLASS_NAMES):
    p = pred == c; t = yva == c
    union = (p | t).sum().item()
    iou = (p & t).sum().item() / union if union else 0
    ious[name] = iou
fig, ax = plt.subplots(figsize=(6, 3.8))
bars = ax.bar(list(ious.keys()), list(ious.values()), color=["#8C8C8C", "#4C72B0", "#DD8452"])
for b, v in zip(bars, ious.values()):
    ax.text(b.get_x()+b.get_width()/2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, 1); ax.set_ylabel("IoU"); ax.set_title(f"U-Net 逐类 IoU（mIoU={sum(ious.values())/len(ious):.4f}）")
plt.tight_layout()
plt.savefig(FIGS / "fig4_perclass.png", dpi=150, bbox_inches="tight")
plt.show()
print({k: f"{v:.4f}" for k, v in ious.items()})

## 6. 总结与下一步

**本项目收获**

1. 分类→分割的范式转移：每像素监督 + mIoU/Dice 评估（边界敏感度）
2. 跳跃连接的量化：U-Net vs FCN 同容量对照，skips 的增益 = 终局 mIoU 差（见 §4）
3. CE+Dice 联合：缓解边界类极不均衡（<5% 像素）的经典配方
4. 分割的可视化语言：掩码叠加与逐类 IoU——与分类的混淆矩阵对应

**下一步**：`02_Detection_torchvision`——检测任务：预训练 Faster R-CNN / RetinaNet 推理，理解 mAP、NMS、Anchor，不做大训练。